# Chuẩn bị môi trường và dữ liệu

In [ ]:
from pathlib import Path
import pandas as pd
import cv2
import numpy as np
from skimage.feature import hog
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.multiclass import OneVsRestClassifier
df = pd.read_csv('tom_and_jerry/ground_truth.csv')

images = []
labels = []


# Loại bỏ viền đen

In [2]:
def bo_vien_den(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    _, thresh = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)

    coords = cv2.findNonZero(thresh)

    if coords is None:
        return image

    x, y, w, h = cv2.boundingRect(coords)

    cropped = image[y:y+h, x:x+w]

    return cropped

# HOG

In [3]:
def extract_hog(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    feature = hog(
        gray,
        orientations=9,
        pixels_per_cell=(8,8),
        cells_per_block=(2,2),
        block_norm='L2-Hys'
    )

    return feature

# RGB Histogram

In [4]:
def extract_rgb_histogram(img):

    hist_b = cv2.calcHist([img], [0], None, [32], [0, 256]).flatten()
    hist_g = cv2.calcHist([img], [1], None, [32], [0, 256]).flatten()
    hist_r = cv2.calcHist([img], [2], None, [32], [0, 256]).flatten()
    # Scale lại
    hist_b /= hist_b.sum()
    hist_g /= hist_g.sum()
    hist_r /= hist_r.sum()

    feature = np.concatenate([hist_b, hist_g, hist_r])

    return feature

# Feature extraction

In [5]:
def extract_feature(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    img = bo_vien_den(img)
    img = cv2.resize(img, (128,128))

    hog_feature = extract_hog(img)
    rgb_feature = extract_rgb_histogram(img)

    feature = np.concatenate([hog_feature, rgb_feature])

    return feature

In [6]:
import psutil

print("CPU:", psutil.cpu_percent(interval=1), "%")
print("RAM:", psutil.virtual_memory().percent, "%")

CPU: 15.3 %
RAM: 69.3 %


# Extract feature

In [7]:
for i, (_, row) in enumerate(df.iterrows()):
    path = f'tom_and_jerry/DF/{row.filename}'
    img = cv2.imread(path)
    feature = extract_feature(img)

    images.append(feature)
    labels.append([row.tom, row.jerry])
    if i % 500 ==0:
        print(i)

0
500
1000
1500
2000
2500
3000
3500
4000
4500
5000


Đầu ra sẽ là một cái vector có dạng là HOG + RGB ( B + G + R) được lưu vào images, còn label thì đc lưu vào biến labels

# In feature ra file csv (sợ chạy xong nổ máy)

In [11]:
X = np.array(images)
y = np.array(labels)

# Tạo DataFrame từ feature
feature_df = pd.DataFrame(X)

# Thêm label
feature_df['tom'] = y[:, 0]
feature_df['jerry'] = y[:, 1]

# Lưu CSV
feature_df.to_csv('tom_jerry_features.csv', index=False)

print(feature_df.shape)

(5478, 8198)


# Đọc file dữ liệu

In [ ]:
df = pd.read_csv('tom_jerry_features.csv')

X = df.drop(columns=['tom', 'jerry']).values
y = df[['tom', 'jerry']].values

print(X.shape)
print(y.shape)

(5478, 8196)
(5478, 2)


# Chia test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=36
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(4382, 8196)
(1096, 8196)


# Scale lại

In [ ]:


scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:


model = OneVsRestClassifier(
    SVC(kernel='rbf', C=1.0, gamma='scale')
)

model.fit(X_train, y_train)

,"estimator estimator: estimator objectA regressor or a classifier that implements :term:`fit`.When a classifier is passed, :term:`decision_function` will be usedin priority and it will fallback to :term:`predict_proba` if it is notavailable.When a regressor is passed, :term:`predict` is used.",SVC()
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation: the `n_classes`one-vs-rest problems are computed in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: 0.20 `n_jobs` default changed from 1 to None",None
,"verbose verbose: int, default=0The verbosity level, if non zero, progress messages are printed.Below 50, the output is sent to stderr. Otherwise, the output is sentto stdout. The frequency of the messages increases with the verbositylevel, reporting all iterations at 10. See :class:`joblib.Parallel` formore details... versionadded:: 1.1",0
Name,Type,Value
"classes_ classes_: array, shape = [`n_classes`]Class labels.","ndarray[int64](2,)","[0,1]"
estimators_ estimators_: list of `n_classes` estimatorsEstimators used for predictions.,list,"[SVC(), SVC()]"
label_binarizer_ label_binarizer_: LabelBinarizer objectObject used to transform multiclass labels to binary labels andvice-versa.,LabelBinarizer,LabelBinarize...e_output=True)
multilabel_ multilabel_: booleanWhether a OneVsRestClassifier is a multilabel classifier.,bool,True
n_classes_ n_classes_: intNumber of classes.,int,2
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 0.24,int,8196
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0


In [19]:
y_pred = model.predict(X_test)

In [20]:
from sklearn.metrics import classification_report

print(classification_report(
    y_test,
    y_pred,
    target_names=['Tom', 'Jerry']
))

              precision    recall  f1-score   support

         Tom       0.91      0.88      0.90       559
       Jerry       0.88      0.70      0.78       415

   micro avg       0.90      0.80      0.85       974
   macro avg       0.90      0.79      0.84       974
weighted avg       0.90      0.80      0.85       974
 samples avg       0.60      0.59      0.59       974



c:\Users\Crink\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Crink\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Crink\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(aver